In [1]:
import getpass
import pandas as pd
import psycopg2 
import json
import requests
from datetime import datetime
import numpy as np

# Variables
# weekly_payout = 15.51

# Define PostgreSQL database connection parameters
# user = input("username")

api_key = getpass.getpass("Enter DataGolf API key:")
host = getpass.getpass("Enter Database Host:")
port = "5432" # The default port for PosgreSQL Server
dbname = 'postgres'
user = getpass.getpass("Enter Username:")
password = getpass.getpass("Enter Password:")

# Define a SQLAlchemy URI string for connecting to the database
# The URI structure is [DB_FLAVOR]+[DB_PYTHON_LIBRARY]://[USERNAME]:[PASSWORD]@[DB_HOST]:[PORT]/[DB_NAME]
db_URI = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"

In [2]:
def df_from_json(feed):
    response = requests.get(feed)
    json_data = response.json()
    df = pd.DataFrame(json_data)
    return df

# Single Event

#### Determine Event ID

In [ ]:
#find event
event_name = 'The American Express'
q = f"""select * from event where event_name = '{event_name}';"""
df_event = pd.read_sql(sql=q, con=db_URI)
df_event

,id_event,calendar_year,event_id,date,event_name,dfs_payout
0,24457dcb-3246-47dd-b795-0062c4397581,2020,2,2020-01-19,The American Express,NaN
1,f1484c4c-f846-47ce-b287-d6a9817c58ca,2021,2,2021-01-24,The American Express,NaN
2,ed09c7d1-ac66-4425-a1f6-70a2cba7e73c,2022,2,2022-01-23,The American Express,NaN
3,59e44a4e-f802-435f-b631-70dee936325c,2023,2,2023-01-22,The American Express,NaN
4,a269f919-d509-4407-8f58-97091d4c8c84,2024,2,2024-01-21,The American Express,NaN
5,51c38193-b4c8-4d9e-8823-0e317a55d1ef,2025,2,2025-01-19,The American Express,15.51


In [ ]:
#grab ID
event_id = df_event['event_id'].values[0]

#### Gather possible courses

In [50]:
q = f"""
select DISTINCT course.course_name from round
inner join course on round.id_course = course.id_course
inner join event on round.id_event = event.id_event
where event.event_name = '{event_name}' and event.calendar_year = 2020;
"""
df_courses = pd.read_sql(sql=q, con=db_URI)
df_courses

,course_name
0,La Quinta CC
1,Nicklaus Tournament Course
2,Stadium Course


#### Gather event history

In [54]:
q = f"""
select event.event_name, 
        event.date, 
        player.player_name, 
        player.dg_id,
        dfs.total_pts, 
        dfs.salary,
        dfs.hole_score_pts,
        dfs.finish_pts,
        dfs.five_birdie_pts,
        dfs.bogey_free_pts,
        dfs.bounce_back_pts,
        dfs.streak_pts
from dfs_total as dfs
inner join event on event.id_event = dfs.id_event
inner join player on player.dg_id = dfs.dg_id
where event.event_id = {event_id};
"""
df_dfs_historical = pd.read_sql(q, db_URI)
df_dfs_historical.sample(3)

,event_name,date,player_name,dg_id,total_pts,salary,hole_score_pts,finish_pts,five_birdie_pts,bogey_free_pts,bounce_back_pts,streak_pts
693,The American Express,2021-01-24,"Landry, Andrew",14000,54.8,NaN,53.9,0,0,0,0.3,0.6
330,The American Express,2023-01-22,"Mitchell, Keith",17365,121.2,9100.0,93.9,3,16,5,0.9,2.4
736,The American Express,2021-01-24,"Taylor, Ben",19948,27.9,NaN,26.7,0,0,0,0.6,0.6


#### Gather upcoming event info

In [49]:
upcoming_json = df_from_json('https://feeds.datagolf.com/field-updates?tour=pga&file_format=json&key='+api_key)
df_dfs_upcoming = pd.DataFrame(pd.json_normalize(upcoming_json['field']))
df_dfs_upcoming.sample(3)
# upcoming_json

,am,country,course,dg_id,dk_id,dk_salary,early_late,fd_id,fd_salary,flag,pga_number,player_name,r1_teetime,r2_teetime,r3_teetime,r4_teetime,start_hole,unofficial,yh_id,yh_salary
6,0,USA,Pete Dye Stadium Course,12333,41588292,6000,1,125790-77869,7000,USA,22764,"Block, Michael",2026-01-22 09:25,2026-01-23 08:52,2026-01-24 10:42,None,10,0,golf.p.9165$golf.g.20263,20.0
9,0,USA,La Quinta Country Club,32152,41588259,6300,0,125790-201440,7100,USA,66417,"Brown, Blades",2026-01-22 09:58,2026-01-23 09:25,2026-01-24 08:52,None,1,0,golf.p.25854$golf.g.20263,20.0
84,0,CHN,La Quinta Country Club,15310,41588186,7300,1,125790-78439,8900,CHN,35296,"Li, Haotong",2026-01-22 08:52,2026-01-23 10:42,2026-01-24 10:09,None,1,0,golf.p.11595$golf.g.20263,20.0


In [ ]:
#ensure length of golfers is in the ballpark
#(might need to cross-check with courses *see above)
len(df_dfs_upcoming['dg_id'].unique())

156

In [ ]:
#filter
df_dfs_upcoming = df_dfs_upcoming[['dg_id', 'fd_salary', 'player_name']]
df_dfs_upcoming.rename(columns={'fd_salary':'salary'}, inplace=True)

In [ ]:
#write to csv
# df_dfs_historical.to_csv('historical.csv', index=False)

# df_dfs_upcoming.to_csv('upcoming.csv', index=False)

# All Events

#### DFS totals

In [3]:
q = f"""
select event.event_name, 
        event.id_event,
        event.date, 
        player.player_name, 
        player.dg_id,
        dfs.total_pts, 
        dfs.salary,
        dfs.hole_score_pts,
        dfs.finish_pts,
        dfs.five_birdie_pts,
        dfs.bogey_free_pts,
        dfs.bounce_back_pts,
        dfs.streak_pts
from dfs_total as dfs
inner join event on event.id_event = dfs.id_event
inner join player on player.dg_id = dfs.dg_id
where date > '2020-01-01';
"""
df_dfs_historical = pd.read_sql(q, db_URI)
df_dfs_historical.sample(10)

,event_name,id_event,date,player_name,dg_id,total_pts,salary,hole_score_pts,finish_pts,five_birdie_pts,bogey_free_pts,bounce_back_pts,streak_pts
15364,Charles Schwab Challenge,2ae733ce-e2a4-4e62-9a09-6bdcb6da221a,2022-05-29,"Munoz, Sebastian",20770,45.6,9000.0,43.2,0,0,0,0.6,1.8
18824,Shriners Children's Open,fd27d2e1-9b7d-4d84-a308-042fe7042328,2021-10-10,"Leishman, Marc",7649,135.6,NaN,95.3,18,16,0,0.3,6.0
10349,Masters Tournament,aeac6db2-2bb5-4592-a2f4-37301849ed5e,2023-04-09,"Fleetwood, Tommy",12294,46.1,9100.0,43.6,1,0,0,0.9,0.6
15898,Mexico Open at Vidanta,79d8f2ff-97a8-4f21-8535-5e5170f62e9e,2022-05-01,"Kitayama, Kurt",21891,124.4,8100.0,88.8,20,12,0,0.6,3.0
29137,Valero Texas Open,0e770c7a-3347-4db5-8635-9045ffea4e6b,2025-04-06,"Martin, Ben",14007,46.1,8200.0,44.6,0,0,0,0.3,1.2
31841,TOUR Championship,e7dac4e1-93f7-4633-8deb-fc01329d66e8,2025-08-24,"Fleetwood, Tommy",12294,145.4,12100.0,92.2,30,16,0,1.2,6.0
25004,Corales Puntacana Resort & Club Championship,e1632502-849a-43fa-880e-0b5f8b1d6260,2020-09-27,"Hickok, Kramer",21237,76.0,NaN,66.0,3,4,0,0.6,2.4
25666,Wyndham Championship,f399d768-4248-44c4-9206-61f4170dd5d7,2020-08-16,"Kizzire, Patton",13470,81.3,NaN,70.6,0,8,0,0.9,1.8
5439,Puerto Rico Open,a3e009c4-6da8-42e0-b1a6-39dc247f6928,2024-03-10,"Biondi, Fred",25614,81.1,NaN,71.2,1,8,0,0.3,0.6
33187,Farmers Insurance Open,f44aff21-82f0-4f05-ad9e-3dca367936b2,2026-02-01,"Villegas, Camilo",9171,12.7,7000.0,12.4,0,0,0,0.3,0.0


#### Round data

In [4]:
q = f"""
select round.*, player.player_name, course.course_name, event.event_name, event.date
from round
inner join player on round.dg_id = player.dg_id
inner join course on round.id_course = course.id_course
inner join event on round.id_event = event.id_event
where event.date > '2020-01-01';
"""
df_round_historical = pd.read_sql(q, db_URI)
# df_round_historical.drop(columns=['id_course'], inplace=True)
df_round_historical.sample(10)

,id_round,id_event,id_course,dg_id,round,start_hole,teetime,score,pars,birdies,...,scrambling,prox_rgh,prox_fw,gir,driving_dist,driving_acc,player_name,course_name,event_name,date
37165,b097d34f-2a43-474c-981a-4b12a9efa674,b434da5c-8e0e-4e77-a156-513d035dd572,6e2d922c-1691-4e8d-bc60-f319d0ac7faf,6762,2.0,10,8:53am,72.0,14.0,2.0,...,0.500,49.109,36.219,0.667,267.9,0.500,"Perez, Pat",TPC Sawgrass,THE PLAYERS Championship,2022-03-13
7534,7ff8783a-e27d-4a46-9ed3-2db1a370678d,07967d3f-7374-47f1-bc3b-381fec3d74b5,0cf13c95-939a-4583-ab21-2b6b05119085,7399,2.0,1,7:30am,75.0,9.0,4.0,...,0.429,45.602,19.491,0.722,291.4,0.500,"Glover, Lucas",TPC Twin Cities,3M Open,2020-07-26
51559,639b65d5-9d1a-40cc-81fd-d70d382d5301,e019f147-1bb7-497b-91f3-5a77dfd560ac,0c14204f-307a-43ce-b438-050a79b70e30,12423,1.0,10,7:30am,64.0,10.0,7.0,...,1.000,33.761,24.182,0.889,288.2,0.643,"Kirk, Chris",Waialae CC,Sony Open in Hawaii,2023-01-15
13679,36b247c9-582c-4dca-9a08-162dd9b3e123,07c8bf76-a9bd-472d-8858-2aeb285c5263,42322c75-b29d-4bec-adc5-4c838b100e92,18103,4.0,10,9:06am,71.0,11.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,"Bezuidenhout, Christiaan",Augusta National GC,The Masters,2020-11-15
50360,e7ca7a0e-a22e-4914-8907-f4b98d98eb42,131d1be7-cec6-46b9-839c-6a7b1e144089,7f2b425c-b0f9-4f19-af03-232e9b227c17,13404,1.0,10,1:25pm,68.0,13.0,4.0,...,NaN,NaN,NaN,0.667,310.5,0.643,"Gligic, Michael",El Camaleon GC,World Wide Technology Championship at Mayakoba,2022-11-06
13910,0b13bd07-e901-4207-a798-746b6473196b,6b1efa67-16af-4063-99bd-75272a753b4c,d80cc616-e822-42bc-9a3a-fae4362eaf74,5994,4.0,10,10:25am,63.0,11.0,7.0,...,0.333,27.012,25.976,0.944,272.5,0.857,"Stenson, Henrik",Sea Island Resort (Seaside),The RSM Classic,2020-11-22
46118,44b4a8ce-908c-4d4b-bc8d-9081892d0e00,7f5eee25-7825-4e0d-8c1b-43c8c6bafb00,689582fd-0d30-41a6-89a8-344350374c90,16715,2.0,10,12:32pm,69.0,13.0,4.0,...,0.600,37.381,26.806,0.833,304.2,0.571,"Ryder, Sam",Detroit Golf Club,Rocket Mortgage Classic,2022-07-31
5168,ec3fc43a-2a0a-4510-834c-a7242438f559,fa04868a-4157-4898-9d09-f4b399b8e0ab,fa1be5e0-7034-4248-946c-49afe94ce935,5665,1.0,1,8:13am,68.0,10.0,6.0,...,0.400,14.805,34.865,0.833,278.6,0.786,"Cink, Stewart",Harbour Town GL,RBC Heritage,2020-06-21
89972,8c9bd8b6-71ab-493c-aac6-d2cc7d026b30,36a20d88-7188-40c5-9137-381137606cfe,1014af43-84dd-469b-bb05-42e75af17c20,13872,4.0,1,9:21am,73.0,10.0,2.0,...,0.500,95.339,41.791,0.500,300.6,0.714,"Bradley, Keegan",Torrey Pines GC (South),Farmers Insurance Open,2025-01-25
25038,721913f5-d2bf-4dd2-921c-7ac46aaf0ec0,013911f4-1a7e-4a17-8118-3ff787f8cdd4,689582fd-0d30-41a6-89a8-344350374c90,13831,2.0,1,7:55am,65.0,9.0,8.0,...,NaN,11.549,14.936,1.000,277.0,0.857,"Knox, Russell",Detroit Golf Club,Rocket Mortgage Classic,2021-07-04


#### upcoming df

In [5]:
upcoming_json = df_from_json('https://feeds.datagolf.com/field-updates?tour=pga&file_format=json&key='+api_key)
df_dfs_upcoming = pd.DataFrame(pd.json_normalize(upcoming_json['field']))
df_dfs_upcoming = df_dfs_upcoming[['dg_id', 'fd_salary', 'player_name']]
df_dfs_upcoming.rename(columns={'fd_salary':'salary'}, inplace=True)
df_dfs_upcoming.sample(3)

,dg_id,salary,player_name
66,21943,7000.0,"Lebioda, Hank"
45,23602,9300.0,"Hojgaard, Nicolai"
26,11676,8100.0,"Finau, Tony"


#### download csv

In [6]:
df_dfs_historical.to_csv('historical_dfs_all.csv', index=False)
df_round_historical.to_csv('historical_round_all.csv', index=False)
df_dfs_upcoming.to_csv('upcoming.csv', index=False)

#### Post-event score compilation